In [ ]:
!pip install catboost lightgbm xgboost pygeohash optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

import pygeohash as pgh

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [ ]:
print("Duplicate rows:", train.duplicated().sum())

Duplicate rows: 0


In [ ]:
target = train["demand"]

test_ids = test["Index"]

train = train.drop("demand", axis=1)

full = pd.concat([train, test], axis=0).reset_index(drop=True)

print(full.shape)

(119077, 10)


In [ ]:
def decode_geohash(g):

    try:
        lat, lon = pgh.decode(g)
        return pd.Series([lat, lon])

    except:
        return pd.Series([np.nan, np.nan])

full[["latitude", "longitude"]] = full["geohash"].apply(
    decode_geohash
)

In [ ]:
full["RoadType_missing"] = full["RoadType"].isnull().astype(int)

full["Weather_missing"] = full["Weather"].isnull().astype(int)

full["Temperature_missing"] = (
    full["Temperature"].isnull().astype(int)
)

In [ ]:
full["RoadType"] = full["RoadType"].fillna(
    full["RoadType"].mode()[0]
)

full["Weather"] = full["Weather"].fillna(
    full["Weather"].mode()[0]
)

full["Temperature"] = full["Temperature"].fillna(
    full["Temperature"].median()
)

In [ ]:
full["hour"] = (
    full["timestamp"]
    .str.split(":")
    .str[0]
    .astype(int)
)

full["minute"] = (
    full["timestamp"]
    .str.split(":")
    .str[1]
    .astype(int)
)

In [ ]:
full["hour_sin"] = np.sin(
    2*np.pi*full["hour"]/24
)

full["hour_cos"] = np.cos(
    2*np.pi*full["hour"]/24
)

full["minute_sin"] = np.sin(
    2*np.pi*full["minute"]/60
)

full["minute_cos"] = np.cos(
    2*np.pi*full["minute"]/60
)

In [ ]:
full["rush_hour"] = (
    full["hour"].isin([7,8,9,17,18,19])
).astype(int)

In [ ]:
for col in [
    "geohash",
    "RoadType",
    "Weather"
]:

    freq = full[col].value_counts()

    full[col + "_freq"] = (
        full[col].map(freq)
    )

In [ ]:
full["lat_lon"] = (
    full["latitude"]
    *
    full["longitude"]
)

full["lat_sq"] = (
    full["latitude"]**2
)

full["lon_sq"] = (
    full["longitude"]**2
)

In [ ]:
full["lane_temp"] = (
    full["NumberofLanes"]
    *
    full["Temperature"]
)

full["hour_temp"] = (
    full["hour"]
    *
    full["Temperature"]
)

full["lane_hour"] = (
    full["NumberofLanes"]
    *
    full["hour"]
)

In [ ]:
from sklearn.cluster import KMeans

for k in [10,20,50]:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    full[f'cluster_{k}'] = kmeans.fit_predict(
        full[['latitude','longitude']]
    )

In [ ]:
full['geo3'] = full['geohash'].astype(str).str[:3]
full['geo4'] = full['geohash'].astype(str).str[:4]
full['geo5'] = full['geohash'].astype(str).str[:5]

In [ ]:
for col in ['geo3','geo4','geo5']:

    freq = full[col].value_counts()

    full[col+'_freq'] = full[col].map(freq)

In [ ]:
full['temp_hour'] = (
    full['Temperature']
    *
    full['hour']
)

full['temp_lane'] = (
    full['Temperature']
    *
    full['NumberofLanes']
)

full['hour_lane'] = (
    full['hour']
    *
    full['NumberofLanes']
)

full['lat_hour'] = (
    full['latitude']
    *
    full['hour']
)

full['lon_hour'] = (
    full['longitude']
    *
    full['hour']
)

In [ ]:
full["distance_center"] = np.sqrt(
    (full["latitude"] - full["latitude"].mean())**2 +
    (full["longitude"] - full["longitude"].mean())**2
)

In [ ]:
full["temp_bin"] = pd.cut(
    full["Temperature"],
    bins=10,
    labels=False
)

In [ ]:
full["lane_cat"] = (
    full["NumberofLanes"]
    .astype(str)
)

In [ ]:
full["peak_period"] = (
    (
        full["hour"].between(7,9)
    ) |
    (
        full["hour"].between(17,19)
    )
).astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [

    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",

    "geo3",
    "geo4",
    "geo5",

    "cluster_10",
    "cluster_20",
    "cluster_50",
    "temp_bin",
    "lane_cat"

]

for col in cat_cols:

    le = LabelEncoder()

    full[col] = le.fit_transform(
        full[col].astype(str)
    )

In [ ]:
train_processed = full.iloc[:len(target)]

test_processed = full.iloc[len(target):]

In [ ]:
features = train_processed.columns.tolist()

features.remove("Index")
features.remove("timestamp")

X = train_processed[features]

y = target

X_test = test_processed[features]

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
cat_oof = np.zeros(len(X))

cat_preds = np.zeros(len(X_test))

for train_idx, valid_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        loss_function="RMSE",
        eval_metric="R2",
        verbose=0
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_valid)

    cat_oof[valid_idx] = pred

    cat_preds += (
        model.predict(X_test) / 5
    )

print(
    "CatBoost CV:",
    r2_score(y, cat_oof)
)

CatBoost CV: 0.9489483369149693


In [ ]:
lgb_oof = np.zeros(len(X))

lgb_preds = np.zeros(len(X_test))

for train_idx, valid_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    model = LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    lgb_oof[valid_idx] = pred

    lgb_preds += (
        model.predict(X_test)/5
    )

print(
    "LightGBM CV:",
    r2_score(y, lgb_oof)
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010735 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4184
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 43
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033029 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4185
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 43
[LightGBM] [Info] Start training from score 0.093797
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.054237 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4187
[LightGBM] [Info] Number of data points in the train set

In [ ]:
xgb_oof = np.zeros(len(X))

xgb_preds = np.zeros(len(X_test))

for train_idx, valid_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    model = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    xgb_oof[valid_idx] = pred

    xgb_preds += (
        model.predict(X_test)/5
    )

print(
    "XGBoost CV:",
    r2_score(y, xgb_oof)
)

XGBoost CV: 0.9514557622916999
